# Module 2 Cleanup: Snowflake Postgres Portal

This notebook removes all objects created by Module 2 (`hol-module2.ipynb`) and restores the EPOWER Agent to its Module 1 state.

**Execution order matters — dependencies require this sequence:**

| Step | Where | What |
|------|-------|------|
| 1 | **Postgres** (psql) | Drop tables, pipelines, extensions |
| 2 | **Snowflake** (this notebook) | Drop data objects (Iceberg table, views, stage) |
| 3 | **Snowflake** (this notebook) | Drop catalog integration |
| 4 | **Snowflake** (this notebook) | Detach network policy from Postgres instance |
| 5 | **Snowflake** (this notebook) | Drop Postgres instance |
| 6 | **Snowflake** (this notebook) | Drop network policy and rule |
| 7 | **Snowflake** (this notebook) | Restore agent to Module 1 state |

> **Why this order?** The network policy cannot be dropped while it is still assigned to the Postgres instance. You must first detach the policy (`ALTER POSTGRES INSTANCE ... UNSET NETWORK_POLICY`), then drop the instance, and only then drop the policy. Attempting to drop the policy before detaching will fail with a "cannot be dropped as it is associated with one or more entities" error.

## Step 1: Postgres Cleanup (run in psql)

Run this in your Postgres client **before** continuing:

```bash
psql service=my_epower_portal -f cleanup-module2-postgres.sql
```

Or manually:

```sql
SELECT incremental.drop_pipeline('sync_portal_activity_to_iceberg');
DROP TABLE IF EXISTS portal_activity_log_iceberg;
DROP TABLE IF EXISTS portal_activity_log CASCADE;
DROP TABLE IF EXISTS service_requests CASCADE;
DROP TABLE IF EXISTS tariff_orders CASCADE;
DROP TABLE IF EXISTS meter_readings CASCADE;
DROP TABLE IF EXISTS portal_users CASCADE;
DROP EXTENSION IF EXISTS pg_incremental CASCADE;
DROP EXTENSION IF EXISTS pg_cron CASCADE;
DROP EXTENSION IF EXISTS pg_lake CASCADE;
```

Once done, continue with Step 2 below.

## Step 2: Drop Data Objects & Catalog Integration

Remove tables, views, and the catalog integration created by Module 2.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

-- Drop data objects
DROP ICEBERG TABLE IF EXISTS EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;
DROP TABLE IF EXISTS EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT;
DROP SEMANTIC VIEW IF EXISTS EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW;
DROP STAGE IF EXISTS EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEED_STAGE;

-- Drop catalog integration
DROP CATALOG INTEGRATION IF EXISTS PORTAL_POSTGRES_CATALOG;

In [ ]:
%%sql
-- Step 3: Detach network policy from Postgres instance
-- (MUST happen BEFORE dropping the policy — a policy cannot be dropped
--  while still assigned to an entity)
BEGIN
    ALTER POSTGRES INSTANCE MY_EPOWER_PORTAL UNSET NETWORK_POLICY;
EXCEPTION
    WHEN OTHER THEN NULL;  -- Instance may not exist
END;

-- Step 4: Drop Postgres instance (irreversible)
DROP POSTGRES INSTANCE IF EXISTS MY_EPOWER_PORTAL;

-- Step 5: Drop network objects (now safe — policy is detached)
USE DATABASE EPOWER_DEMO;
DROP NETWORK POLICY IF EXISTS EPOWER_PG_POLICY;
DROP NETWORK RULE IF EXISTS EPOWER_PG_INGRESS;

## Step 6: Restore Agent to Module 1 State

The Module 2 notebook added `portal_analyst` to the EPOWER Agent. This cell recreates the agent **without** the portal tool.

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;

CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms, your explanatory text and insights MUST be in the user's language.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export → vpp_telemetry_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## Verification

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

SHOW POSTGRES INSTANCES LIKE 'MY_EPOWER%';
SHOW CATALOG INTEGRATIONS LIKE 'PORTAL%';
SHOW NETWORK POLICIES LIKE 'EPOWER_PG%';

---

Module 2 cleanup complete. The EPOWER demo is back to Module 1 state.

Don't forget to remove the psql connection:

```bash
# Remove from ~/.pg_service.conf: [my_epower_portal] section
# Remove from ~/.pgpass: the corresponding line
```